# Who gets the offer? Logistic regression on real bookings

CL1 of *Predictive Analytics*. Work in **pairs**: one types, one writes down the predictions; swap at every station.

**How it works.** The code already runs, you do not have to write it. At every station you

1. **predict** what will happen (write it down *before* you run the cell),
2. **run** the cell and compare,
3. **change** one thing,
4. **decide** and give a reason.

**Read** questions are in the style of the exam: answer them on paper first, then run the cell below.

**Stuck?** Every station only needs the *Setup* cell. Just go on with the next station.

| Station | Question | Time |
|---|---|---|
| 1 | How good is doing nothing? | 8 min |
| 2 | Which features help? | 12 min |
| 3 | Too good to be true? | 6 min |
| 4 | Which threshold? | 10 min |
| 5 | Promise and test | 10 min |
| Extra | For the fast ones | |

> *From: Mara, hotel manager. Attached are 20,000 real bookings. Tomorrow morning I meet the owners. Tell me who gets the prepaid offer and how much profit you promise per 1,000 bookings. I would rather have an honest number than a high one.*

As in the lecture, the offer costs **30 €** and a prevented cancellation is worth **100 €** (fictional). Data: bookings of two hotels in Portugal, 2015 to 2017 (Antonio, Almeida, Nunes 2019, [Hotel booking demand datasets](https://doi.org/10.1016/j.dib.2018.11.126), CC BY 4.0), cleaned, a random sample.

## Setup

Run this cell once (about 10 seconds). You do not have to read the code.

In [ ]:
#@title Setup (run me) { display-mode: "form" }
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://fjoelsak.github.io/thm_wk-1609-predictiveanalytics/cl1-who-gets-the-offer/hotel-bookings.csv"
FILE = Path("hotel-bookings.csv")
df = pd.read_csv(FILE if FILE.exists() else URL)

# 60 % train, 20 % validation (for decisions), 20 % test (only for the last station)
train, rest = train_test_split(df, test_size=0.4, random_state=0, stratify=df["cancelled"])
valid, _test = train_test_split(rest, test_size=0.5, random_state=0, stratify=rest["cancelled"])

OFFER_COST, SAVED = 30, 100
CATEGORICAL = ["hotel", "month", "meal", "channel", "customer_type", "room_type"]
NUMERIC = [c for c in df.columns if c not in CATEGORICAL + ["cancelled", "parking_spaces"]]
ALL = NUMERIC + CATEGORICAL


def profit_per_1000(y, offered):
    """Profit per 1,000 bookings: +70 for each offer to a canceller, -30 for each other offer."""
    y, offered = np.asarray(y), np.asarray(offered, dtype=bool)
    return 1000 * np.mean(offered * (SAVED * y - OFFER_COST))


def report(y, offered):
    """Confusion matrix and measures for a list of offers."""
    tn, fp, fn, tp = confusion_matrix(y, offered, labels=[0, 1]).ravel()
    print(pd.DataFrame([[tn, fp], [fn, tp]], index=["came", "cancelled"], columns=["no offer", "offer"]), "\n")
    print(f"sensitivity {tp / (tp + fn):.2f}   specificity {tn / (tn + fp):.2f}   "
          f"precision {tp / (tp + fp) if tp + fp else float('nan'):.2f}   accuracy {(tp + tn) / len(y):.2f}")
    print(f"profit per 1,000 bookings: {profit_per_1000(y, offered):.0f} €")


def check(name, value, expected, tolerance=0.01):
    """Compare your value with the expected one."""
    ok = value is not None and abs(value - expected) <= tolerance
    print(f"{name}:", "correct ✅" if ok else "not yet ❌ (have another look at the output above)")


def hand_in(y, offered, decision, promise):
    """Last station: report on the test data and print the line to hand in."""
    report(y, offered)
    profit = profit_per_1000(y, offered)
    print(f"\nHAND IN >>> {decision} | promised {promise:.0f} € | reached {profit:.0f} € "
          f"| score {profit - abs(profit - promise):.0f}")


def train_model(features):
    """Logistic regression on the given features, with scaling and one-hot encoding."""
    num = [f for f in features if f not in CATEGORICAL]
    cat = [f for f in features if f in CATEGORICAL]
    prepare = make_column_transformer((StandardScaler(), num), (OneHotEncoder(drop="first"), cat))
    return make_pipeline(prepare, LogisticRegression(max_iter=2000)).fit(train[features], train["cancelled"])


def auc(features):
    """AUC on the validation data of a model with the given features."""
    return roc_auc_score(valid["cancelled"], train_model(features).predict_proba(valid[features])[:, 1])


def final_test(features, threshold, promise):
    """Station 5: evaluate the decision on new bookings."""
    new = _test.assign(parking_spaces=0)  # nobody has a parking space yet when the booking arrives
    offered = train_model(features).predict_proba(new[features])[:, 1] >= threshold
    hand_in(_test["cancelled"], offered, f"threshold {threshold}", promise)


print(f"{len(train)} train, {len(valid)} validation, {len(_test)} test bookings")
df.head()

## Station 1: How good is doing nothing?

**Predict:** about 27 % of the bookings were cancelled. What accuracy does a "model" reach that offers the rate to **nobody**? What profit? And what happens if **everybody** gets the offer?

**Prediction:** *write here*

In [ ]:
print("Offer to nobody")
report(valid["cancelled"], np.zeros(len(valid)))
print("\nOffer to everybody")
report(valid["cancelled"], np.ones(len(valid)))

**Change:** the manager thinks about a smaller discount. Up to which discount does the offer to *everybody* pay off? Try values for `cost`, then enter the limit in `break_even_cost`.

In [ ]:
cost = 30  # discount in €, change me
print(f"profit per 1,000 bookings if everybody gets the offer: {1000 * (SAVED * valid['cancelled'].mean() - cost):.0f} €")

break_even_cost = None  # your result in €
check("Break-even", break_even_cost, SAVED * valid["cancelled"].mean(), tolerance=0.6)

**Read:** which values does `gain` contain, and what is `gain.mean()`?

```python
y       = np.array([1, 0, 1, 0])      # 1 = cancelled
offered = np.array([True, True, False, False])
gain    = offered * (100 * y - 30)
```

**Answer:** *write here*

In [ ]:
y = np.array([1, 0, 1, 0])
offered = np.array([True, True, False, False])
gain = offered * (100 * y - 30)
print(gain, gain.mean())

**Decide:** a colleague is proud of a model with 75 % accuracy. What do you answer, in one sentence?

**Answer:** *write here*

## Station 2: Which features help?

**Predict:** from these six features, pick the **three** that predict cancellations best. Write them down before you run anything.

| Feature | Meaning |
|---|---|
| `lead_time` | days between booking and arrival |
| `channel` | portal, direct, agency, corporate, group, other |
| `customer_type` | Transient (single booking), Transient-Party (linked to others), Group, Contract |
| `earlier_cancellations` | how often the guest cancelled before |
| `price` | average price per night in € |
| `month` | month of arrival |

**Prediction:** *write here*

In [ ]:
my_three = ["price", "month", "earlier_cancellations"]  # change me

CANDIDATES = ["lead_time", "channel", "customer_type", "earlier_cancellations", "price", "month"]
better = sum(auc(list(c)) > auc(my_three) + 1e-9 for c in combinations(CANDIDATES, 3))
print(f"AUC of your three features: {auc(my_three):.3f}   (rank {better + 1} of the 20 possible combinations, 1 = best)")
print(f"AUC of all {len(ALL)} features:   {auc(ALL):.3f}")

How does the model use the features? The table shows the **odds factors** $e^b$ of the lecture, the strongest first. Above 1: more cancellations, below 1: fewer, close to 1: hardly any effect.

- **Categories** are compared with the one that is left out (`channel`: agency, `customer_type`: Contract, `month`: April). Example: if `channel_portal` is 3.1, the odds of a portal booking are about three times the odds of an agency booking, everything else being equal.
- **Numbers** are scaled: the factor refers to one standard deviation more.

In [ ]:
m2 = train_model(CANDIDATES)
names = [n.split("__")[1] for n in m2[0].get_feature_names_out()]
factors = pd.Series(np.exp(m2[-1].coef_[0]), index=names, name="odds factor")
factors.sort_values(key=lambda f: -abs(np.log(f))).round(2).to_frame()

**Change:** which features do the rows at the top of the table belong to? (Several rows can belong to the same feature.) Use them to improve your three features above.

**Read:** `channel` has six categories, but the table shows only five rows `channel_...`. Which one is missing, and why? Then: which shape does `encoded` have, and which shape without `drop="first"`?

```python
bookings = pd.DataFrame({"channel": ["portal", "direct", "agency", "portal"]})
encoded  = OneHotEncoder(drop="first").fit_transform(bookings)
```

**Answer:** *write here*

In [ ]:
bookings = pd.DataFrame({"channel": ["portal", "direct", "agency", "portal"]})
print(OneHotEncoder(drop="first").fit_transform(bookings).shape, OneHotEncoder().fit_transform(bookings).shape)

**Decide:** would you use your three features or all of them? Give one reason for each choice.

**Answer:** *write here*

## Station 3: Too good to be true?

A colleague found one more column: `parking_spaces`, the number of parking spaces of the booking. With it, the model gets clearly better.

**Predict:** 27 % of all bookings are cancelled. What share do you expect among the bookings **with** a parking space?

**Prediction:** *write here*

In [ ]:
with_parking = ALL + ["parking_spaces"]
print(f"AUC without parking: {auc(ALL):.3f}")
print(f"AUC with parking:    {auc(with_parking):.3f}")

parking = np.where(train["parking_spaces"] > 0, "with parking space", "without parking space")
for group, rows in train.groupby(parking):
    print(f"{group}: {rows['cancelled'].mean():.1%} of {len(rows)} bookings cancelled")

Not a single cancellation: suspicious. You call the reception. *"The parking space? We enter it when the guest **arrives**."* So whoever cancels never gets an entry. And on the day of the booking, when you have to decide about the offer, the column is **0 for everybody**.

**Change:** here is a guest who booked on a portal 135 days ahead and came by car. Run the cell (today's entry: 1), then set `parking = 0` (what you knew on the day of the booking) and run it again. What happens to the probability and to your decision?

In [ ]:
parking = 1  # change me to 0: the entry on the day of the booking

with_parking = ALL + ["parking_spaces"]
guest = valid[valid["parking_spaces"] > 0].iloc[[11]].assign(parking_spaces=parking)
p = train_model(with_parking).predict_proba(guest[with_parking])[0, 1]
print(f"parking_spaces = {parking}: probability of a cancellation {p:.0%} -> {'offer' if p >= 0.3 else 'no offer'}")

And if the hotel goes live with the colleague's model? Compare the three rows (all validation bookings, threshold 0.3):

In [ ]:
with_parking = ALL + ["parking_spaces"]
cases = {
    "without parking": (ALL, valid),
    "with parking, old data (column filled in)": (with_parking, valid),
    "with parking, new bookings (column is 0)": (with_parking, valid.assign(parking_spaces=0)),
}
result = {}
for name, (features, data) in cases.items():
    p = train_model(features).predict_proba(data[features])[:, 1]
    result[name] = {"AUC": round(roc_auc_score(valid["cancelled"], p), 3), "profit per 1,000 in €": round(profit_per_1000(valid["cancelled"], p >= 0.3))}
pd.DataFrame(result).T.astype({"profit per 1,000 in €": int})

**Decide:** which row tells the hotel what it will really earn? Do you use the column?

**Answer:** *write here*

## Station 4: Which threshold?

**Predict:** at which threshold will the profit be highest on the validation data?

**Prediction:** *write here*

In [ ]:
m3 = train_model(ALL)
p3 = m3.predict_proba(valid[ALL])[:, 1]

thresholds = np.arange(0, 1.01, 0.01)
profits = [profit_per_1000(valid["cancelled"], p3 >= t) for t in thresholds]
plt.plot(thresholds, profits, color="green", linewidth=3)
plt.axhline(0, color="gray", linewidth=1)
plt.xlabel("threshold")
plt.ylabel("profit per 1,000 bookings in €")
plt.show()

**Change:** try at least three thresholds and write down sensitivity, precision, accuracy and profit for each.

In [ ]:
threshold = 0.5  # change me

p3 = train_model(ALL).predict_proba(valid[ALL])[:, 1]
report(valid["cancelled"], p3 >= threshold)

**Read:** what do the two lines print?

```python
p = np.array([0.15, 0.30, 0.45, 0.80])
print((p >= 0.3).sum())
print((p >= 0.5).astype(int))
```

**Answer:** *write here*

In [ ]:
p = np.array([0.15, 0.30, 0.45, 0.80])
print((p >= 0.3).sum())
print((p >= 0.5).astype(int))

**Decide:** which threshold do you recommend? Which measure gets worse because of it, and to whom would you have to explain that?

**Answer:** *write here*

## Station 5: Promise and test

Choose your features and your threshold, and **promise** Mara a profit per 1,000 bookings. `final_test` uses bookings that nobody has seen so far. You have **one shot**.

**Score = profit − |profit − promise|.** You need a good decision *and* an honest promise: promising too much costs points, promising too little as well.

In [ ]:
my_features = ALL
my_threshold = 0.5  # your decision
my_promise = 0      # your promise in € per 1,000 customers

# final_test(my_features, my_threshold, my_promise)   # remove the first # when you are sure

**Hand in:** copy the line `HAND IN >>> ...` and add **one sentence** on why you chose this threshold.

**Decide:** your promise and the result differ. Is the model worse than you thought, or was it chance?

**Answer:** *write here*

## Extra: for the fast ones

1. **Limited capacity:** Mara only allows 100 offers on the validation data. Who gets them? How many of these bookings are really cancelled?
2. **Not everybody accepts:** assume only half of the guests who would cancel accept the offer (and the others do not cost the discount). Does the threshold change? And if the discount has to be paid in every case?
3. **Exam question:** explain in two sentences why the AUC does not change when you change the threshold.

In [ ]:
# your space